# 005: RNN Mathematical Recurrence — Hidden state dynamics, vanishing gradients, and from-scratch RNN

## RECURRENCE EQUATION
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$y_t = W_{hy} h_t + b_y$$
- The hidden state $h_t$ acts as a **memory** that accumulates information from all previous time steps.
- **Vanishing Gradient**: During BPTT, gradients are multiplied by $W_{hh}$ at each step. If $\|W_{hh}\| < 1$, gradients shrink exponentially.
---


In [ ]:
import numpy as np

class VanillaRNN:
    """From-scratch RNN for sequence processing."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        scale = 0.01
        self.Wxh = np.random.randn(hidden_dim, input_dim) * scale
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * scale
        self.Why = np.random.randn(output_dim, hidden_dim) * scale
        self.bh = np.zeros((hidden_dim, 1))
        self.by = np.zeros((output_dim, 1))
        self.hidden_dim = hidden_dim

    def forward(self, inputs):
        """Forward pass over a sequence. inputs: list of column vectors."""
        h = np.zeros((self.hidden_dim, 1))
        self.hidden_states = [h]
        outputs = []
        for x in inputs:
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            y = self.Why @ h + self.by
            self.hidden_states.append(h)
            outputs.append(y)
        return outputs



In [ ]:
print("--- RNN Forward Pass ---")
np.random.seed(42)
rnn = VanillaRNN(input_dim=3, hidden_dim=4, output_dim=2)

# Sequence of 5 time steps, each with 3 features
sequence = [np.random.randn(3, 1) for _ in range(5)]
outputs = rnn.forward(sequence)

for t, (out, h) in enumerate(zip(outputs, rnn.hidden_states[1:])):
    print(f"t={t} | h norm={np.linalg.norm(h):.4f} | output={out.flatten().round(4)}")



In [ ]:
# ── Vanishing gradient demonstration ──
print("\n--- Gradient Magnitude Across Time Steps ---")
h = np.zeros((4, 1))
grad_norms = []
for t in range(50):
    h = np.tanh(rnn.Whh @ h + np.random.randn(4, 1) * 0.1)
    # Approximate gradient: product of Jacobians
    diag = 1 - h.flatten()**2  # tanh derivative
    jacobian_norm = np.linalg.norm(rnn.Whh * diag.reshape(1, -1))
    grad_norms.append(jacobian_norm)

print(f"Jacobian norm at t=1:  {grad_norms[0]:.6f}")
print(f"Jacobian norm at t=25: {grad_norms[24]:.6f}")
print(f"Jacobian norm at t=49: {grad_norms[48]:.6f}")
print("[Insight] If Jacobian norm < 1 consistently, gradients vanish exponentially.")
